In [1]:
import time
import pandas as pd
import ast
import wandb.apis.public
import itertools
import concurrent.futures
import tqdm
from pathlib import Path
import json
import os
import sys
sys.path.append("..")

import wandb
os.environ["WANDB_SILENT"] = "true" # Do not print download output.
api = wandb.Api(timeout=100)

In [2]:
root_sweeps = Path("../../wandb-sweep-data-qnrl/")
root_sweeps.mkdir(parents=True, exist_ok=True)

In [3]:
entity_id = "news-vt" # Team name in WandB.
project_id = "qnrl" # Project ID in WandB.

In [4]:
# List of sweep names and their IDs for reference.
sweep_name_id = { # name: id
    # # QnRL.
    # 'qnrl/FrozenLake-v1': '7yoeyqqr',
    # 'qnrl/CliffWalking-v1': 'nin93kq8',
    # 'qnrl/Acrobot-v1': '5ehazde3',
    # 'qnrl/CartPole-v1': 'qojis1fw',
    # 'qnrl/Breakout-v5': 'eigl7wk5',
    # 'qnrl/SpaceInvaders-v5': '8adr9d1k',

    # # C51.
    # 'c51/FrozenLake-v1': 'jzduwzvg',
    # 'c51/CliffWalking-v1': '8xyt1eb5',
    # 'c51/Acrobot-v1': 'n8imxz1v',
    # 'c51/CartPole-v1': 'arv708xs',
    # 'c51/Breakout-v5': 'dg4sodfc',
    # 'c51/SpaceInvaders-v5': 'wbmr06xp',
    
    # qC51.
    # 'qc51/FrozenLake-v1': 'n93n0nr1',
    # 'qc51/CliffWalking-v1': 'gicedw73',
    # 'qc51/Acrobot-v1': 'h0z9k7g5',
    # 'qc51/CartPole-v1': 'foiafm3w',
    'qc51/Breakout-v5': 'sya2ty4z',
    'qc51/SpaceInvaders-v5': 'vapn48ds',
}

In [5]:
def get_run_config_dict(run: wandb.apis.public.Run):
    """Compute run config hash from original run program command string."""
    # Download the run metadata file, load as JSON dictionary, and then delete the local downloaded file.
    metadata_file = run.file("wandb-metadata.json").download(replace=True)
    metadata = json.load(metadata_file)
    if os.path.exists(metadata_file.name):
        os.remove(metadata_file.name)
    
    # Get config string from run program command arguments.
    # Format is: --config="{...}"
    cfg_str = metadata['args'][1].split("--config=")[1]
    # Convert this string to dictionary via literal evaluation.
    cfg_dict = ast.literal_eval(cfg_str)
    return cfg_dict

In [6]:
import shutil
from typing import Literal
import numpy as np
import copy


def download_run_data(
    root: Path,
    run: wandb.apis.public.Run,
    replace_existing: bool = False,
    skip_cache: bool = False,
    history_source: Literal['artifact', 'scan'] = 'artifact',
    scan_page_size: int = 100000,
    ):
    """Download files for the given run. Returns a list of functions that when called will download the associated file."""
    root_run = root / f"run-{str(run.id)}"
    root_run.mkdir(parents=True, exist_ok=True)
    
    # TODO download all files for a run.
    # list(api.run("zanderman-virginia-tech/qnrl/o8bvmgah").files())
    
    def _download_file(file):
        """Download a single file by its WandB file name."""
        try:
            if not (root_run/file.name).exists() or (replace_existing and (root_run/file.name).exists()):
                file.download(root=str(root_run), replace=replace_existing, exist_ok=True)
        except Exception as e:
            print(f"[{run.id} - file:{file.name}] error: {e}")
    
    def _gen_multidownload_files():
        """Return a generator to single-file downloader functions."""
        return (
            lambda file=file: _download_file(file)
            for file in run.files()
        )

    # def _download_config():
    #     """Download run config."""
    #     try:
    #         file_name = "config.yaml"
    #         if not (root_run/file_name).exists() or (replace_existing and (root_run/file_name).exists()):
    #             run.file(file_name).download(root=str(root_run), replace=replace_existing, exist_ok=True)
    #     except Exception as e:
    #         print(f"[{run.id} - config] error: {e}")
    
    # def _download_summary():
    #     """Download run summary."""
    #     try:
    #         file_name = "wandb-summary.json"
    #         if not (root_run/file_name).exists() or (replace_existing and (root_run/file_name).exists()):
    #             run.file(file_name).download(root=str(root_run), replace=replace_existing, exist_ok=True)
    #     except Exception as e:
    #         print(f"[{run.id} - summary] error: {e}")
    
    # def _download_metadata():
    #     """Download run metadata."""
    #     try:
    #         file_name = "wandb-metadata.json"
    #         if not (root_run/file_name).exists() or (replace_existing and (root_run/file_name).exists()):
    #             run.file(file_name).download(root=str(root_run), replace=replace_existing, exist_ok=True)
    #     except Exception as e:
    #         print(f"[{run.id} - metadata] error: {e}")
    
    def _download_history():
        """Download run metrics history."""
        try:
            root_history = root_run / "history"
            artifact_kwargs = dict(name=f"{entity_id}/{project_id}/run-{run.id}-history:latest", type="wandb-history")
            
            # See also new `run.download_history_exports()` from v0.24.1+
            if history_source == 'artifact' and api.artifact_exists(**artifact_kwargs):
                history_artifact = api.artifact(**artifact_kwargs)
                if not root_history.exists() or replace_existing:
                    # Remove existing.
                    if root_history.exists(): shutil.rmtree(root_history)
                    # Create new directory.
                    root_history.mkdir(parents=True, exist_ok=True)
                    # Download artifact.
                    history_artifact.download(root=str(root_history), skip_cache=skip_cache)
            else:
                outfile = root_history / "0000.parquet"
                if not outfile.exists() or replace_existing:
                    # print(f"[{run.id}] download via scan_history")
                    df = pd.DataFrame(list(run.scan_history(page_size=scan_page_size)))
                    # Replace known strings with equivalents.
                    df = (df
                        .replace(['Infinity', '+Infinity'], np.inf)
                        .replace('-Infinity', -np.inf)
                        .replace('NaN', np.nan)
                        .infer_objects(copy=False) # Infer column type from objects within it.
                    )
                    
                    # Remove existing.
                    if root_history.exists(): shutil.rmtree(root_history)
                    # Create new directory.
                    root_history.mkdir(parents=True, exist_ok=True)
                    # Save artifact to file.
                    df.to_parquet(outfile)
        except Exception as e:
            print(f"[{run.id} - history] error: {e}")
        
    def _download_models():
        """Download run saved models."""
        try:
            artifact_kwargs = dict(name=f"{entity_id}/{project_id}/run-{run.id}-models:latest", type="model")
            if api.artifact_exists(**artifact_kwargs):
                models_artifact = api.artifact(**artifact_kwargs)
                root_models = root_run / "models"
                if not root_models.exists() or replace_existing:
                    # Remove existing.
                    if root_models.exists(): shutil.rmtree(root_models)
                    # Create new directory.
                    root_models.mkdir(parents=True, exist_ok=True)
                    # Download artifact.
                    models_artifact.download(root=str(root_models), skip_cache=skip_cache)
        except Exception as e:
            print(f"[{run.id} - models] error: {e}")
    
    
    yield from _gen_multidownload_files()
    yield from [
        # _download_config,
        # _download_summary,
        # _download_metadata,
        _download_history,
        _download_models,
    ]

def download_all_run_data_for_sweep(
    root: Path,
    sweep: wandb.apis.public.Sweep,
    replace_existing: bool = False,
    skip_cache: bool = False,
    history_source: Literal['artifact', 'scan'] = 'artifact',
    scan_page_size: int = 100000,
    ):
    """Download files for all runs associated with the given sweep."""
    root_sweep = root / f"sweep-{str(sweep.id)}"
    root_sweep.mkdir(parents=True, exist_ok=True)
    yield from itertools.chain.from_iterable(
        download_run_data(
            root=root_sweep,
            run=run,
            replace_existing=replace_existing,
            skip_cache=skip_cache,
            history_source=history_source,
            scan_page_size=scan_page_size,
        )
        for run in sweep.runs
    )

In [7]:
# ###
# # Download a specific run.
# ###
# # runs_to_download = ['ixfifpwm', 'z3k9i6db', 'nujoh0r7', 'm7w1ciph']
# # runs_to_download = ['ixfifpwm', 'z3k9i6db', 'nujoh0r7', 'm7w1ciph']
# runs_to_download = ['ixfifpwm']
# # root_runs = root_sweeps/f"sweep-sd4va92r"

# files_to_download = list(tqdm.tqdm(
#     itertools.chain.from_iterable(
#         download_run_data(
#             root=root_sweeps / f"sweep-{api.run(f'{project_id}/{run_id}').sweep.id}",
#             run=api.run(f"{project_id}/{run_id}"),
#             replace_existing=True,
#         )
#         for run_id in runs_to_download
#     ),
#     unit="files",
#     desc="Preparing downloads",
# ))

# # Download all files and folders for given sweeps.
# num_threads = min(32, (os.cpu_count() or 1) + 4) # Maximum number of concurrent threads
# with concurrent.futures.ThreadPoolExecutor(max_workers=num_threads) as executor:
#     list(tqdm.tqdm(executor.map(lambda f: f(), files_to_download), total=len(files_to_download), unit="files", desc="Downloading"))

In [8]:
replace_existing = False
skip_cache = True
history_source = 'scan'

###
# Download all sweeps.
###
sweeps_to_download = list(sweep_name_id.values())

# Download all files and folders for given sweeps.
num_threads = min(32, (os.cpu_count() or 1) + 4) # Maximum number of concurrent threads
print(f"{num_threads} threads")
with concurrent.futures.ThreadPoolExecutor(max_workers=num_threads) as executor:
    
    def worker_check_file(sweep_id):
        for item in download_all_run_data_for_sweep(
            root=root_sweeps, 
            sweep=api.sweep(f"{entity_id}/{project_id}/{sweep_id}"),
            replace_existing=replace_existing,
            skip_cache=skip_cache,
            history_source=history_source,
        ):
            yield item
            time.sleep(0.01)
    
    # Collect list of files and folders to download.
    files_to_download = list(tqdm.tqdm(
        itertools.chain.from_iterable(
            executor.map(
                # lambda sweep_id: download_all_run_data_for_sweep(
                #     root=root_sweeps, 
                #     sweep=api.sweep(f"{project_id}/{sweep_id}"),
                #     replace_existing=replace_existing,
                #     skip_cache=skip_cache,
                #     history_source=history_source,
                # ),
                worker_check_file,
                sweeps_to_download
            )
        ),
        unit="files",
        desc="Preparing downloads",
    ))
    
    def worker_download_file(func):
        func()
        time.sleep(0.01) # TODO add rate limiting (WandB is max 200 calls per minute)
    
    # Download the files.
    list(tqdm.tqdm(executor.map(worker_download_file, files_to_download), total=len(files_to_download), unit="files", desc="Downloading"))

18 threads


Preparing downloads: 5204files [02:38, 32.91files/s]
Downloading: 100%|██████████| 5204/5204 [46:38<00:00,  1.86files/s]  
